# step 5 — inspect filtered data & create gold standard sample

this notebook inspects the filtered dataset and draws a random sample for manual gold standard annotation.

**What this notebook does:**
- loads and previews the filtered policy entries
- draws a fixed random sample of 20 entries (seed=42, do not change after initial draw)
- reports word count statistics and document-type / year distributions for both the full dataset and the sample
- saves the sample to `data/gold_standard/`

**Default use case**

German Covid era dataset (`germany_cov_filtered_final.json`)

**USAGE**

Run cells top to bottom. To adapt for a new country or crisis period:
1. Update the filenames in the path definitions (cells below) — `PROJECT_ROOT` and the `data/processed/` subfolder stay the same.
2. Adjust `SAMPLE_SIZE` and `RANDOM_SEED` if needed — but fix them before annotation begins and do not change afterwards.


In [ ]:
# pip installs:


#imports:

##imports for reading data
import json

#imports for word counts
import json
from pathlib import Path

##imports for plots
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from collections import Counter

# ─── PROJECT ROOT ─────────────────────────────────────────────────────────────
# Notebook lives at policy-tracker/2_processing/germany/ — root is 2 levels up.
# All file paths below are built from PROJECT_ROOT; only filenames need to change.
PROJECT_ROOT = Path().resolve().parents[1]


In [ ]:
#load data

file_path = PROJECT_ROOT / "data/processed/germany/germany_cov_filtered_final.json"

with open(file_path, "r", encoding="utf-8") as file:
    data = json.load(file)

entries = data if isinstance(data, list) else list(data.values())
for i, entry in enumerate(entries[:70]):
    print(f"--- Entry {i+1} ---")
    print(json.dumps(entry, ensure_ascii=False, indent=2))

print(f"the dataset contains {len(data)} policies")


In [ ]:
# draw random sample of 20 for gold-standard annotation
# seed is fixed — do not change after the sample is drawn
import random
import json
from pathlib import Path
from datetime import datetime

SAMPLE_SIZE = 20
RANDOM_SEED = 42

random.seed(RANDOM_SEED)
sample = random.sample(data, SAMPLE_SIZE)

OUTPUT_DIR = PROJECT_ROOT / "data/gold_standard"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
timestamp = datetime.now().strftime("%Y%m%d")
output_file = OUTPUT_DIR / f"germany_cov_sample_20_{timestamp}.json"

output_file.write_text(json.dumps(sample, ensure_ascii=False, indent=2), encoding="utf-8")

print(f"Population: {len(data)} policies")
print(f"Sample:     {SAMPLE_SIZE} policies  (seed={RANDOM_SEED})")
print(f"Saved:      {output_file}")
print()
for i, entry in enumerate(sample, 1):
    print(f"  {i:>2}. [{entry.get('similarity_score', 0):.3f}]  {entry['title'][:100]}")


In [ ]:
# word count of full_text in gold-standard sample

sample_path = PROJECT_ROOT / "data/processed/germany/germany_cov_sample_20.json"
sample = json.loads(sample_path.read_text(encoding="utf-8"))

word_counts = [
    (len(entry.get("full_text", "").split()), entry["title"])
    for entry in sample
]

counts_only = [wc for wc, _ in word_counts]

print(f"min:  {min(counts_only):>6} words")
print(f"max:  {max(counts_only):>6} words")
print(f"mean: {sum(counts_only)/len(counts_only):>6.0f} words")
print()
for wc, title in sorted(word_counts):
    print(f"  {wc:>6}  {title[:90]}")


In [ ]:
DATASETS = [
    (PROJECT_ROOT / "data/processed/germany/germany_cov_filtered_final.json", "Full Dataset — Germany COVID-era 2019-2022"),
    (PROJECT_ROOT / "data/processed/germany/germany_cov_sample_20.json",      "Gold Standard Sample — Germany COVID-era 2019-2022"),
]

def plot_overview(records, title):
    word_counts  = [len(e.get("full_text", "").split()) for e in records]
    doc_types    = Counter(e.get("doc_type", "unknown") for e in records)
    years        = Counter(e.get("year") for e in records)
    mean_wc      = sum(word_counts) / len(word_counts)
    sorted_years = sorted(years.items())

    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    fig.suptitle(f"{title}  (n={len(records)})", fontsize=13, fontweight="bold", y=1.02)

    # word count histogram
    ax = axes[0]
    ax.hist(word_counts, bins=8, color="#4C72B0", edgecolor="white", linewidth=0.8)
    ax.axvline(min(word_counts), color="#DD4949", linestyle="--", linewidth=1.2, label=f"min  {min(word_counts):,}")
    ax.axvline(max(word_counts), color="#2CA02C", linestyle="--", linewidth=1.2, label=f"max  {max(word_counts):,}")
    ax.axvline(mean_wc,          color="orange",  linestyle=":",  linewidth=1.4, label=f"mean {mean_wc:,.0f}")
    ax.set_title("Word count (full_text)", fontweight="bold")
    ax.set_xlabel("words"); ax.set_ylabel("documents")
    ax.legend(fontsize=8.5)
    ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{int(x):,}"))

    # doc_type bar chart
    ax = axes[1]
    labels, values = zip(*sorted(doc_types.items(), key=lambda x: -x[1]))
    bars = ax.bar(labels, values, color="#4C72B0", edgecolor="white")
    ax.bar_label(bars, padding=3, fontsize=9)
    ax.set_title("Document type (doc_type)", fontweight="bold")
    ax.set_xlabel("doc_type"); ax.set_ylabel("count")
    ax.set_ylim(0, max(values) + 2)
    plt.setp(ax.get_xticklabels(), rotation=20, ha="right", fontsize=9)

    # year bar chart
    ax = axes[2]
    yr_labels, yr_values = zip(*sorted_years)
    bars = ax.bar([str(y) for y in yr_labels], yr_values, color="#4C72B0", edgecolor="white")
    ax.bar_label(bars, padding=3, fontsize=9)
    ax.set_title("Year distribution (year)", fontweight="bold")
    ax.set_xlabel("year"); ax.set_ylabel("count")
    ax.set_ylim(0, max(yr_values) + 2)

    plt.tight_layout()
    plt.show()

    print(f"\nWord count — min: {min(word_counts):,}   max: {max(word_counts):,}   mean: {mean_wc:,.0f}")
    print(f"\ndoc_type counts:")
    for k, v in sorted(doc_types.items(), key=lambda x: -x[1]):
        print(f"  {k:<25} {v}")
    print(f"\nyear counts:")
    for yr, cnt in sorted_years:
        print(f"  {yr}   {cnt}")


for path, title in DATASETS:
    records = json.loads(path.read_text(encoding="utf-8"))
    if isinstance(records, dict):
        records = list(records.values())
    print(f"\n{'='*60}\n  {title}\n{'='*60}")
    plot_overview(records, title)
